In [ ]:
# Import essential libraries for data manipulation, numerical operations, visualization, and model persistence
import math  # Mathematical operations and constants
import numpy as np  # Numerical computing and array operations
import pandas as pd  # Data manipulation and analysis
import plotly.express as px  # Interactive data visualization
import pickle  # Object serialization for saving/loading models

In [ ]:
# Load the training and testing datasets from CSV files
# These datasets contain simple linear regression data with features (x) and target values (y)
train_data = pd.read_csv('/kaggle/input/datasets/manitejagaddam/simple-linear-regression/train.csv')
test_data = pd.read_csv('/kaggle/input/datasets/manitejagaddam/simple-linear-regression/test.csv')

In [ ]:
# Data preprocessing: Remove any rows with missing values (NaN)
# This ensures data quality and prevents errors during model training
train_data = train_data.dropna()
test_data = test_data.dropna()

,x,y
0,24.0,21.549452
1,50.0,47.464463
2,15.0,17.218656
3,38.0,36.586398
4,87.0,87.288984


# Linear Regression Theory

## Mathematical Foundation

### Hypothesis Function
The linear regression hypothesis is defined as:
$$h(x) = w \cdot x + b$$

Where:
- $h(x)$ is the predicted value
- $w$ is the weight (coefficient)
- $x$ is the input feature
- $b$ is the bias (intercept)

### Cost Function (Mean Squared Error)
$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} (h(x^{(i)}) - y^{(i)})^2$$

Where:
- $m$ is the number of training examples
- $x^{(i)}$ is the i-th training example
- $y^{(i)}$ is the i-th target value

### Gradient Descent Update Rules
$$w = w - \alpha \frac{\partial J}{\partial w} = w - \alpha \frac{1}{m} \sum_{i=1}^{m} (h(x^{(i)}) - y^{(i)}) \cdot x^{(i)}$$

$$b = b - \alpha \frac{\partial J}{\partial b} = b - \alpha \frac{1}{m} \sum_{i=1}^{m} (h(x^{(i)}) - y^{(i)})$$

Where $\alpha$ is the learning rate.

In [ ]:
# Exploratory Data Analysis: Create scatter plot to visualize the relationship between features and target
# This helps understand the data distribution and potential linear relationship
px.scatter(x=train_data['x'], y=train_data['y'], template='seaborn')

# Feature Scaling and Standardization

## Why Standardization Matters

### Z-Score Normalization
Standardization transforms features to have:
- Mean ($\mu$) = 0
- Standard deviation ($\sigma$) = 1

The transformation formula:
$$z = \frac{x - \mu}{\sigma}$$

### Benefits for Gradient Descent

1. **Faster Convergence**: Features on similar scales prevent the algorithm from oscillating
2. **Numerical Stability**: Prevents overflow/underflow in computations
3. **Equal Feature Importance**: Ensures no single feature dominates due to scale

### Important Considerations

- **Data Leakage Prevention**: Always compute $\mu$ and $\sigma$ from training data only
- **Consistent Transformation**: Apply the same transformation to test data using training statistics
- **Impact on Interpretability**: Standardized coefficients represent importance in standard deviations

In [ ]:
# Prepare feature and target variables for model training and testing
# Extract independent variable (x) and dependent variable (y) from datasets
X_train = train_data['x'].values  # Training features
y_train = train_data['y'].values  # Training targets

X_test = test_data['x'].values    # Testing features
y_test = test_data['y'].values    # Testing targets

In [ ]:
def standardize_data(X_train, X_test):
    """
    Standardizes the input data using mean and standard deviation.
    
    Feature scaling is crucial for gradient descent convergence and preventing
    features with large scales from dominating the learning process.

    Parameters:
        X_train (numpy.ndarray): Training data features.
        X_test (numpy.ndarray): Testing data features.

    Returns:
        Tuple of standardized training and testing data.
    """
    # Calculate statistics from training data only (to avoid data leakage)
    mean = np.mean(X_train, axis=0)  # Feature means
    std = np.std(X_train, axis=0)    # Feature standard deviations
    
    # Apply z-score normalization: (x - mean) / std
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std  # Use training stats for test data
    
    return X_train, X_test

# Apply feature scaling to normalize the data
X_train, X_test = standardize_data(X_train, X_test)

# Model Architecture and Training

## Linear Regression Model Structure

### Forward Propagation
The model computes predictions using:
$$\hat{y} = W \cdot X + b$$

Where:
- $\hat{y}$: Predicted values
- $W$: Weight matrix (coefficients)
- $X$: Input features
- $b$: Bias term (intercept)

### Backward Propagation (Gradient Computation)

#### Weight Gradient
$$\frac{\partial J}{\partial W} = \frac{1}{m} X^T \cdot (\hat{y} - y)$$

#### Bias Gradient
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)$$

### Convergence Criteria
Training stops when:
- Parameter updates become smaller than tolerance: $|\theta_{new} - \theta_{old}| < \epsilon$
- Maximum iterations reached
- Cost function stabilizes

### Learning Rate Considerations
- **Too Small**: Slow convergence
- **Too Large**: Oscillation or divergence
- **Just Right**: Steady convergence to optimal solution

In [ ]:
# Reshape feature arrays to be compatible with linear regression model
# Convert from 1D arrays (n_samples,) to 2D arrays (n_samples, n_features)
# This is required because the model expects input shape: [batch_size, num_features]
X_train = np.expand_dims(X_train, axis=-1)  # Shape: (n_samples, 1)
X_test = np.expand_dims(X_test, axis=-1)    # Shape: (n_samples, 1)

In [ ]:
class LinearRegression:
    """
    Linear Regression Model with Gradient Descent

    Linear regression is a supervised machine learning algorithm used for modeling the relationship
    between a dependent variable (target) and one or more independent variables (features) by fitting
    a linear equation to the observed data.

    This class implements a linear regression model using gradient descent optimization for training.
    It provides methods for model initialization, training, prediction, and model persistence.

    Parameters:
        learning_rate (float): The learning rate used in gradient descent.
        convergence_tol (float, optional): The tolerance for convergence (stopping criterion). Defaults to 1e-6.

    Attributes:
        W (numpy.ndarray): Coefficients (weights) for the linear regression model.
        b (float): Intercept (bias) for the linear regression model.

    Methods:
        initialize_parameters(n_features): Initialize model parameters.
        forward(X): Compute the forward pass of the linear regression model.
        compute_cost(predictions): Compute the mean squared error cost.
        backward(predictions): Compute gradients for model parameters.
        fit(X, y, iterations, plot_cost=True): Fit the linear regression model to training data.
        predict(X): Predict target values for new input data.
        save_model(filename=None): Save the trained model to a file using pickle.
        load_model(filename): Load a trained model from a file using pickle.

    Examples:
        >>> from linear_regression import LinearRegression
        >>> model = LinearRegression(learning_rate=0.01)
        >>> model.fit(X_train, y_train, iterations=1000)
        >>> predictions = model.predict(X_test)
    """

    def __init__(self, learning_rate, convergence_tol=1e-6):
        self.learning_rate = learning_rate
        self.convergence_tol = convergence_tol
        self.W = None
        self.b = None

    def initialize_parameters(self, n_features):
        """
        Initialize model parameters.

        Parameters:
            n_features (int): The number of features in the input data.
        """
        self.W = np.random.randn(n_features) * 0.01
        self.b = 0

    def forward(self, X):
        """
        Compute the forward pass of the linear regression model.

        Parameters:
            X (numpy.ndarray): Input data of shape (m, n_features).

        Returns:
            numpy.ndarray: Predictions of shape (m,).
        """
        return np.dot(X, self.W) + self.b

    def compute_cost(self, predictions):
        """
        Compute the mean squared error cost.

        Parameters:
            predictions (numpy.ndarray): Predictions of shape (m,).

        Returns:
            float: Mean squared error cost.
        """
        m = len(predictions)
        cost = np.sum(np.square(predictions - self.y)) / (2 * m)
        return cost

    def backward(self, predictions):
        """
        Compute gradients for model parameters.

        Parameters:
            predictions (numpy.ndarray): Predictions of shape (m,).

        Updates:
            numpy.ndarray: Gradient of W.
            float: Gradient of b.
        """
        m = len(predictions)
        self.dW = np.dot(predictions - self.y, self.X) / m
        self.db = np.sum(predictions - self.y) / m

    def fit(self, X, y, iterations, plot_cost=True):
        """
        Fit the linear regression model to the training data.

        Parameters:
            X (numpy.ndarray): Training input data of shape (m, n_features).
            y (numpy.ndarray): Training labels of shape (m,).
            iterations (int): The number of iterations for gradient descent.
            plot_cost (bool, optional): Whether to plot the cost during training. Defaults to True.

        Raises:
            AssertionError: If input data and labels are not NumPy arrays or have mismatched shapes.

        Plots:
            Plotly line chart showing cost vs. iteration (if plot_cost is True).
        """
        assert isinstance(X, np.ndarray), "X must be a NumPy array"
        assert isinstance(y, np.ndarray), "y must be a NumPy array"
        assert X.shape[0] == y.shape[0], "X and y must have the same number of samples"
        assert iterations > 0, "Iterations must be greater than 0"

        self.X = X
        self.y = y
        self.initialize_parameters(X.shape[1])
        costs = []

        for i in range(iterations):
            predictions = self.forward(X)
            cost = self.compute_cost(predictions)
            self.backward(predictions)
            self.W -= self.learning_rate * self.dW
            self.b -= self.learning_rate * self.db
            costs.append(cost)

            if i % 100 == 0:
                print(f'Iteration: {i}, Cost: {cost}')

            if i > 0 and abs(costs[-1] - costs[-2]) < self.convergence_tol:
                print(f'Converged after {i} iterations.')
                break

        if plot_cost:
            fig = px.line(y=costs, title="Cost vs Iteration", template="plotly_dark")
            fig.update_layout(
                title_font_color="#41BEE9",
                xaxis=dict(color="#41BEE9", title="Iterations"),
                yaxis=dict(color="#41BEE9", title="Cost")
            )

            fig.show()

    def predict(self, X):
        """
        Predict target values for new input data.

        Parameters:
            X (numpy.ndarray): Input data of shape (m, n_features).

        Returns:
            numpy.ndarray: Predicted target values of shape (m,).
        """
        return self.forward(X)
    

    def save_model(self, filename=None):
        """
        Save the trained model to a file using pickle.

        Parameters:
            filename (str): The name of the file to save the model to.
        """
        model_data = {
            'learning_rate': self.learning_rate,
            'convergence_tol': self.convergence_tol,
            'W': self.W,
            'b': self.b
        }

        with open(filename, 'wb') as file:
            pickle.dump(model_data, file)

    @classmethod
    def load_model(cls, filename):
        """
        Load a trained model from a file using pickle.

        Parameters:
            filename (str): The name of the file to load the model from.

        Returns:
            LinearRegression: An instance of the LinearRegression class with loaded parameters.
        """
        with open(filename, 'rb') as file:
            model_data = pickle.load(file)

        # Create a new instance of the class and initialize it with the loaded parameters
        loaded_model = cls(model_data['learning_rate'], model_data['convergence_tol'])
        loaded_model.W = model_data['W']
        loaded_model.b = model_data['b']

        return loaded_model

In [ ]:
# Initialize and train the linear regression model
# Learning rate of 0.01 controls how quickly the model learns
# 10,000 iterations allow sufficient training for convergence
lr = LinearRegression(0.01)
lr.fit(X_train, y_train, 10000)

In [ ]:
# Save the trained model to disk for future use
# This allows us to reuse the model without retraining
lr.save_model('model.pkl')

# Model Evaluation Metrics

## Regression Performance Measures

### 1. Mean Squared Error (MSE)
$$\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

**Interpretation:**
- Average squared difference between predicted and actual values
- Penalizes large errors more heavily (quadratic penalty)
- Units: squared units of target variable
- Lower values indicate better performance

### 2. Root Mean Squared Error (RMSE)
$$\text{RMSE} = \sqrt{\frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2}$$

**Interpretation:**
- Square root of MSE
- Same units as target variable (more interpretable)
- Represents typical prediction error magnitude
- Useful for understanding practical significance

### 3. R-squared (Coefficient of Determination)
$$R^2 = 1 - \frac{\sum_{i=1}^{m} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{m} (y_i - \bar{y})^2}$$

**Interpretation:**
- Proportion of variance in target explained by features
- Range: 0 to 1 (or negative for very poor models)
- $R^2 = 0.85$ means 85% of variance is explained
- Higher values indicate better model fit

# Key Observations and Insights

## Data Characteristics
- **Linear Relationship**: The scatter plot suggests a strong linear correlation between x and y
- **Data Quality**: No missing values after preprocessing, ensuring clean training data
- **Feature Scaling**: Standardization ensures optimal gradient descent performance

## Model Performance Analysis

### Expected Results
- **High R²**: Given the apparent linear relationship, we expect $R^2 > 0.9$
- **Low RMSE**: Should be small relative to the range of y values
- **Convergence**: With learning rate 0.01 and 10,000 iterations, model should converge

### Practical Implications
- **Interpretability**: Linear models provide clear coefficient interpretation
- **Extrapolation Risk**: Predictions outside training range may be unreliable
- **Assumptions**: Model assumes linear relationship and homoscedasticity

## Potential Improvements
1. **Cross-validation**: More robust performance estimation
2. **Regularization**: Prevent overfitting with L1/L2 regularization
3. **Feature Engineering**: Polynomial features for non-linear relationships
4. **Hyperparameter Tuning**: Optimize learning rate and iterations

# Mathematical Derivation Details

## Gradient Descent Derivation

### Cost Function Expansion
Starting from MSE cost function:
$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} (wx^{(i)} + b - y^{(i)})^2$$

### Partial Derivative with Respect to w
$$\frac{\partial J}{\partial w} = \frac{1}{2m} \sum_{i=1}^{m} 2(wx^{(i)} + b - y^{(i)}) \cdot x^{(i)}$$

$$\frac{\partial J}{\partial w} = \frac{1}{m} \sum_{i=1}^{m} (h(x^{(i)}) - y^{(i)}) \cdot x^{(i)}$$

### Partial Derivative with Respect to b
$$\frac{\partial J}{\partial b} = \frac{1}{2m} \sum_{i=1}^{m} 2(wx^{(i)} + b - y^{(i)})$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (h(x^{(i)}) - y^{(i)})$$

## Closed-Form Solution (Normal Equation)
For comparison, the analytical solution is:
$$\theta = (X^T X)^{-1} X^T y$$

**Advantages of Gradient Descent:**
- Works for very large datasets
- Computationally efficient for high-dimensional data
- Can be extended to non-linear models

In [ ]:
# Load the saved model from disk
# This demonstrates model persistence and reusability
model = LinearRegression.load_model("model.pkl")

In [ ]:
class RegressionMetrics:
    """
    Utility class for evaluating regression model performance.
    
    Provides common regression metrics to assess how well the model
    predicts target values compared to actual values.
    """
    
    @staticmethod
    def mean_squared_error(y_true, y_pred):
        """
        Calculate the Mean Squared Error (MSE).
        
        MSE measures the average squared difference between predicted and actual values.
        Lower values indicate better model performance.

        Args:
            y_true (numpy.ndarray): The true target values.
            y_pred (numpy.ndarray): The predicted target values.

        Returns:
            float: The Mean Squared Error.
        """
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = np.mean((y_true - y_pred) ** 2)
        return mse

    @staticmethod
    def root_mean_squared_error(y_true, y_pred):
        """
        Calculate the Root Mean Squared Error (RMSE).
        
        RMSE is the square root of MSE, providing error in the same units as the target.
        More interpretable than MSE for understanding prediction accuracy.

        Args:
            y_true (numpy.ndarray): The true target values.
            y_pred (numpy.ndarray): The predicted target values.

        Returns:
            float: The Root Mean Squared Error.
        """
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = RegressionMetrics.mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        return rmse

    @staticmethod
    def r_squared(y_true, y_pred):
        """
        Calculate the R-squared (R²) coefficient of determination.
        
        R² measures the proportion of variance in the target variable that is
        predictable from the features. Values range from 0 to 1, with higher
        values indicating better model fit.

        Args:
            y_true (numpy.ndarray): The true target values.
            y_pred (numpy.ndarray): The predicted target values.

        Returns:
            float: The R-squared (R²) value.
        """
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mean_y = np.mean(y_true)
        ss_total = np.sum((y_true - mean_y) ** 2)      # Total sum of squares
        ss_residual = np.sum((y_true - y_pred) ** 2)   # Residual sum of squares
        r2 = 1 - (ss_residual / ss_total)
        return r2

In [ ]:
# Model Evaluation: Assess the trained model's performance on test data
# This step is crucial to understand how well the model generalizes to unseen data

# Generate predictions on the test dataset
y_pred = model.predict(X_test)

# Calculate various regression metrics to quantify model performance
mse_value = RegressionMetrics.mean_squared_error(y_test, y_pred)
rmse_value = RegressionMetrics.root_mean_squared_error(y_test, y_pred)
r_squared_value = RegressionMetrics.r_squared(y_test, y_pred)

# Display evaluation results
print(f"Mean Squared Error (MSE): {mse_value}")
print(f"Root Mean Squared Error (RMSE): {rmse_value}")
print(f"R-squared (Coefficient of Determination): {r_squared_value}")